# MA Crossover + MA200 Trend Filter — Backtest

> Variant: Trend Filter (spec 7.2) | Base: MA_CROSSOVER_V1
> Logic: Golden Cross is only valid when price > MA200

## How it works

The base MA crossover generates a golden cross whenever MA_fast crosses above MA_slow.
In a downtrend, many of these golden crosses are false — they are dead-cat bounces.

The MA200 trend filter adds one simple rule: **only buy when price is above MA200**.
This eliminates all golden crosses that occur during bear markets.

## Pipeline

1. Load Data -> 2. Calculate MA(fast) + MA(slow) + MA200
3. Generate signals: golden cross only counts if price > MA200
4. Run backtest with costs -> 5. Metrics -> 6. Visualize

## Controls

- CHOSEN: switch stocks (603986.SH / 002594.SZ / 688981.SH / HK variants)
- FAST_PERIOD / SLOW_PERIOD: tune MA parameters
- MA200_PERIOD: trend filter period (default 200)
- ENABLE_FILTER: toggle trend filter on/off to see the difference

In [ ]:
# ============================================================
# 1. Parameters
# ============================================================
FAST_PERIOD     = 10
SLOW_PERIOD     = 60
MA200_PERIOD    = 200       # Trend filter period
MA_TYPE         = "ema"
ENABLE_FILTER   = True      # True = with MA200 trend filter, False = base strategy

INITIAL_CAPITAL = 1_000_000
COMMISSION_RATE = 0.0003
STAMP_TAX_RATE  = 0.0005
SLIPPAGE_RATE   = 0.001

print(f"Strategy: {MA_TYPE.upper()}({FAST_PERIOD})/{MA_TYPE.upper()}({SLOW_PERIOD})")
print(f"Trend Filter: {'ON (price > MA200)' if ENABLE_FILTER else 'OFF (base strategy)'}")
print(f"Costs: comm={COMMISSION_RATE:.3%} stamp={STAMP_TAX_RATE:.3%} slip={SLIPPAGE_RATE:.1%}")

In [ ]:
# ============================================================
# 2. Load Data
# ============================================================
import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings('ignore')

PROJECT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(PROJECT, "data")

STOCKS = {}
for dname in sorted(os.listdir(DATA_DIR)):
    dpath = os.path.join(DATA_DIR, dname)
    if not os.path.isdir(dpath): continue
    for fname in ['daily_adjusted.csv', 'daily_hk.csv']:
        fp = os.path.join(dpath, fname)
        if os.path.exists(fp):
            code = dname.split('_')[0]
            label = f"{code} ({'A' if 'adjusted' in fname else 'HK'})"
            STOCKS[label] = fp

CHOSEN = list(STOCKS.keys())[2]  # Default: 603986.SH (GigaDevice)

df = pd.read_csv(STOCKS[CHOSEN], encoding="utf-8-sig", parse_dates=["trade_date"])
df = df.sort_values("trade_date").reset_index(drop=True)
col = "close_qfq" if "close_qfq" in df.columns else "close"
df["price"] = df[col]
print(f"Stock: {CHOSEN} | Rows: {len(df)} | {df.trade_date.min().date()} ~ {df.trade_date.max().date()}")

In [ ]:
# ============================================================
# 3. Moving Averages (including MA200)
# ============================================================
def calc_ma(s, n, t):
    return s.ewm(span=n, adjust=False).mean() if t == "ema" else s.rolling(n).mean()

df["ma_fast"] = calc_ma(df["price"], FAST_PERIOD, MA_TYPE)
df["ma_slow"] = calc_ma(df["price"], SLOW_PERIOD, MA_TYPE)
df["ma200"]   = calc_ma(df["price"], MA200_PERIOD, "sma")  # MA200 always SMA

# Warmup: only drop the minimum needed for fast/slow MAs.
# MA200 will have NaN for the first 200 bars -- that's fine for chart display.
# When filter is ON, bars with NaN MA200 are treated as "filter not available, safe to skip".
warmup = max(FAST_PERIOD, SLOW_PERIOD)
df_v = df.iloc[warmup:].copy().reset_index(drop=True)

fast_lbl = f"{MA_TYPE.upper()}({FAST_PERIOD})"
slow_lbl = f"{MA_TYPE.upper()}({SLOW_PERIOD})"
print(f"Warmup: {warmup} bars | Valid: {len(df_v)} rows")
print(f"MA200 latest: {df_v['ma200'].iloc[-1]:.2f}")
print(f"Price vs MA200: {df_v['price'].iloc[-1]:.2f} {'>' if df_v['price'].iloc[-1] > df_v['ma200'].iloc[-1] else '<'} {df_v['ma200'].iloc[-1]:.2f}")

In [ ]:
# ============================================================
# 4. Signal Generation (with MA200 trend filter)
# ============================================================
# Base cross signal
df_v["signal_raw"] = (df_v["ma_fast"] > df_v["ma_slow"]).astype(int)
df_v["cross_raw"]  = df_v["signal_raw"].diff()

# Trend filter: price must be above MA200 for golden cross.
# When MA200 is NaN (not enough history), accept the signal.
if ENABLE_FILTER:
    df_v["trend_ok"] = ((df_v["price"] > df_v["ma200"]) | df_v["ma200"].isna()).astype(int)
    df_v["signal"] = (df_v["signal_raw"] & df_v["trend_ok"]).astype(int)
else:
    df_v["signal"] = df_v["signal_raw"]

df_v["cross"]    = df_v["signal"].diff()
df_v["position"] = df_v["signal"].shift(1).fillna(0).astype(int)

n_g_raw = int((df_v["cross_raw"] == 1).sum())
n_g     = int((df_v["cross"] == 1).sum())
n_d     = int((df_v["cross"] == -1).sum())
print(f"Raw golden crosses: {n_g_raw}")
print(f"Filtered golden crosses: {n_g} (filtered out {n_g_raw - n_g})")
print(f"Death crosses: {n_d} | Trades: {n_g}")

# Show filtered signals
changes = df_v[df_v["cross_raw"] != 0].copy()
changes["raw_label"] = changes["cross_raw"].map({1: "RAW Golden", -1: "RAW Death"})
changes["filtered"]  = df_v.loc[changes.index, "cross"].map({1: "Valid", -1: "Death", 0: "FILTERED"})
changes[["trade_date","price","ma200","raw_label","filtered"]].head(15)

In [ ]:
# ============================================================
# 5. Backtest
# ============================================================
df_v["price_ret"] = df_v["price"].pct_change()
df_v["gross_ret"] = df_v["position"] * df_v["price_ret"]
df_v["trade_act"] = df_v["position"].diff().abs()
df_v["cost"]      = df_v["trade_act"] * (COMMISSION_RATE + SLIPPAGE_RATE)
df_v.loc[df_v["position"].diff() < 0, "cost"] += STAMP_TAX_RATE
df_v["strat_ret"] = df_v["gross_ret"] - df_v["cost"]
df_v["equity"]    = (1 + df_v["strat_ret"].fillna(0)).cumprod()
df_v["bench_eq"]  = (1 + df_v["price_ret"].fillna(0)).cumprod()
pk_s = df_v["equity"].cummax(); pk_b = df_v["bench_eq"].cummax()
df_v["dd"]       = (df_v["equity"] - pk_s) / pk_s
df_v["bench_dd"] = (df_v["bench_eq"] - pk_b) / pk_b
print(f"Strategy: {df_v.equity.iloc[-1]:.4f} ({df_v.equity.iloc[-1]-1:+.2%})")
print(f"Benchmark: {df_v.bench_eq.iloc[-1]:.4f} ({df_v.bench_eq.iloc[-1]-1:+.2%})")

In [ ]:
# ============================================================
# 6. Performance Metrics
# ============================================================
ret = df_v["strat_ret"].dropna(); N = len(ret)
total_ret = df_v["equity"].iloc[-1] - 1
bench_ret = df_v["bench_eq"].iloc[-1] - 1
ann_ret   = (1+total_ret)**(252/N)-1
ann_bench = (1+bench_ret)**(252/N)-1
rc = ret[ret!=0] if len(ret[ret!=0])>0 else ret
sharpe = np.sqrt(252)*rc.mean()/rc.std() if rc.std()>0 else 0
max_dd = df_v["dd"].min(); max_dd_b = df_v["bench_dd"].min()
calmar = ann_ret/abs(max_dd) if max_dd!=0 else 0

g_entry = df_v[df_v["cross"]==1]
trades_ret = []
for i in range(len(g_entry)):
    ei = df_v[df_v["trade_date"]==g_entry["trade_date"].iloc[i]].index[0]
    xi = len(df_v)-1 if i+1>=len(g_entry) else df_v[df_v["trade_date"]==g_entry["trade_date"].iloc[i+1]].index[0]-1
    trades_ret.append(df_v.loc[xi,"price"]/df_v.loc[ei,"price"]-1)
wins=[r for r in trades_ret if r>0]; loses=[r for r in trades_ret if r<=0]
wr=len(wins)/len(trades_ret) if trades_ret else 0
aw=np.mean(wins) if wins else 0
al=np.mean([abs(r) for r in loses]) if loses else 0
plr=aw/al if al>0 else float("inf")

from IPython.display import display, HTML
display(HTML(f"""<table style="border-collapse:collapse;width:100%;font-size:14px">
<tr style="background:#f8f9fa;font-weight:bold"><td style="padding:10px 16px">Metric</td><td style="padding:10px 16px;text-align:right">Filtered Strategy</td><td style="padding:10px 16px;text-align:right">Benchmark</td></tr>
<tr><td>Total Return</td><td style="text-align:right;color:{'#D85A30' if total_ret>0 else '#1D9E75'}">{total_ret:+.2%}</td><td style="text-align:right;color:{'#D85A30' if bench_ret>0 else '#1D9E75'}">{bench_ret:+.2%}</td></tr>
<tr style="background:#f8f9fa"><td>Annual Return</td><td style="text-align:right">{ann_ret:+.2%}</td><td style="text-align:right">{ann_bench:+.2%}</td></tr>
<tr><td>Sharpe (ann)</td><td style="text-align:right;font-weight:bold">{sharpe:.2f}</td><td style="text-align:right">-</td></tr>
<tr style="background:#f8f9fa"><td>Max Drawdown</td><td style="text-align:right;color:#D85A30">{max_dd:+.2%}</td><td style="text-align:right;color:#D85A30">{max_dd_b:+.2%}</td></tr>
<tr><td>Calmar Ratio</td><td style="text-align:right;font-weight:bold">{calmar:.2f}</td><td style="text-align:right">-</td></tr>
<tr style="background:#f8f9fa"><td>Win Rate</td><td style="text-align:right">{wr:.1%}</td><td style="text-align:right">-</td></tr>
<tr><td>P/L Ratio</td><td style="text-align:right;font-weight:bold">{plr:.2f}</td><td style="text-align:right">-</td></tr>
<tr style="background:#f8f9fa"><td>Total Trades</td><td style="text-align:right">{len(trades_ret)}</td><td style="text-align:right">-</td></tr>
</table>"""))

In [ ]:
# ============================================================
# 7. Visualization
# ============================================================
import matplotlib.pyplot as plt, matplotlib.dates as mdates
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(3,1,figsize=(16,12),gridspec_kw={"height_ratios":[2,1.5,1]})
dts = df_v["trade_date"]

ax=axes[0]
ax.plot(dts, df_v["price"], "#B4B2A9", lw=0.8, alpha=0.7, label="Close")
ax.plot(dts, df_v["ma_fast"], "#D85A30", lw=1.2, label=fast_lbl)
ax.plot(dts, df_v["ma_slow"], "#534AB7", lw=1.2, label=slow_lbl)
ax.plot(dts, df_v["ma200"], "#378ADD", lw=1.5, ls="dotted", alpha=0.7, label="MA200")
ax.fill_between(dts, 0, df_v["price"].max()*1.2, where=(df_v["price"]>df_v["ma200"]), alpha=0.05, color="#378ADD")
g=df_v[df_v["cross"]==1]; d=df_v[df_v["cross"]==-1]
ax.scatter(g["trade_date"], g["price"], c="#1D9E75", marker="^", s=80, zorder=5, label="Golden (filtered)")
ax.set_title(f"{CHOSEN} - Price & MA + MA200 Filter", fontsize=14, fontweight="bold")
ax.legend(loc="upper left", fontsize=8); ax.set_ylabel("Price"); ax.grid(True,alpha=0.3)

ax=axes[1]
ax.fill_between(dts, 1, df_v["equity"], alpha=0.1, color="#378ADD")
ax.plot(dts, df_v["equity"], "#378ADD", lw=1.8, label="Filtered Strategy")
ax.plot(dts, df_v["bench_eq"], "#B4B2A9", lw=1.2, ls="dashed", label="Benchmark")
ax.axhline(1, c="black", lw=0.5, ls="dotted")
ax.set_title("Equity Curve", fontsize=14, fontweight="bold")
ax.legend(loc="upper left", fontsize=9); ax.set_ylabel("Equity"); ax.grid(True,alpha=0.3)

ax=axes[2]
ax.fill_between(dts, df_v["dd"], 0, alpha=0.3, color="#D85A30")
ax.fill_between(dts, df_v["bench_dd"], 0, alpha=0.15, color="#B4B2A9")
ax.plot(dts, df_v["dd"], "#D85A30", lw=1, label="Strategy DD")
ax.plot(dts, df_v["bench_dd"], "#B4B2A9", lw=0.8, ls="dashed", label="Benchmark DD")
ax.axhline(0, c="black", lw=0.3); ax.axhline(-0.25, c="#D85A30", lw=0.5, ls="dotted", alpha=0.5)
ax.set_title("Drawdown", fontsize=14, fontweight="bold")
ax.legend(loc="lower left", fontsize=9); ax.set_ylabel("Drawdown"); ax.grid(True,alpha=0.3)
for a in axes:
    a.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    a.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(a.xaxis.get_majorticklabels(), rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8. Summary
# ============================================================
filter_label = "ON (price > MA200)" if ENABLE_FILTER else "OFF (base strategy)"
print(f"""
{'='*60}
  MA Crossover + Trend Filter Backtest: {CHOSEN}
  {MA_TYPE.upper()}({FAST_PERIOD})/{MA_TYPE.upper()}({SLOW_PERIOD}) with MA200 filter
  Filter: {filter_label}
  Return: {total_ret:+.2%} | Sharpe: {sharpe:.2f} | MaxDD: {max_dd:+.2%} | WinRate: {wr:.1%}
  Trades: {len(trades_ret)} | P/L Ratio: {plr:.2f}
{'='*60}

Set ENABLE_FILTER=False to compare with base strategy.
Switch CHOSEN to change stocks.
Based on ma_crossover_strategy_spec.md variant 7.2
""")